# Feature experiments: what is worth adding?

Companion to `per_generator_xgboost.ipynb` (14 per-generator XGBoost models,
193 features, micro AUC 0.9802 on the 2022 validation). This notebook
documents the search for what to add on top of that feature set:

- **five candidate feature groups**, tested one at a time
- **two early-stopping protocols**: stop on the validation set (the old,
  leaky way) vs stop on a holdout slice of the train period
- **two validation years**: 2021 and 2022

Each experiment row trains 14 XGBoost models (~2 minutes). The full grid below
is about 23 rows, so expect roughly 45-60 minutes. Progress is printed with
elapsed seconds; results accumulate in one DataFrame at the end.

In [1]:
import time

import numpy as np
import pandas as pd
import xgboost as xgb
import yaml
from sklearn.metrics import roc_auc_score

HOURS = 168
DATA = "data/kernel"
EXT = "data/extended"


In [2]:
price = pd.read_csv(f"{DATA}/Historical_day_ahead_price_2015_2025.csv")
inflow = pd.read_csv(f"{DATA}/Historical_inflow_1958_2025.csv")
volume = pd.read_csv(f"{DATA}/Historical_volume_2015_2024.csv")
water = pd.read_csv(f"{DATA}/Synthetic_water_value_2015_2024.csv")
target = pd.read_csv(f"{DATA}/Unit_commitment_decisions.csv")
min_flow = pd.read_csv(f"{EXT}/Constraint_min_flow.csv")
min_volume = pd.read_csv(f"{EXT}/Constraint_min_volume.csv")
topology = yaml.safe_load(open(f"{EXT}/Tokke_Vinje_topology.yaml"))

In [3]:
def indexed(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"], utc=True, format="mixed")
    return df.sort_values("date").set_index("date")


price_h = indexed(price)["Day-ahead price (EUR/MWh)"].resample("1h").mean()
inf_wide = indexed(inflow)
vol_wide = indexed(volume)
wv_wide = indexed(water)
mf_wide = indexed(min_flow)
mv_wide = indexed(min_volume)

RESERVOIRS = list(vol_wide.columns)
INF_COLS = list(inf_wide.columns)
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}

## Which reservoirs sit around each generator?

From the topology YAML: the intake reservoirs feeding each plant (through
tunnels), the reservoir downstream of each plant, and each plant's outlet
line. The new candidate features are built from these.

In [4]:
connections = topology["connections"]


def supplying_reservoirs(name, kind):
    if kind == "reservoir":
        return {name}
    out = set()
    for e in connections:
        if e["to"] == name and e["to_type"] == kind and e["from_type"] in {"reservoir", "tunnel"}:
            out |= supplying_reservoirs(e["from"], e["from_type"])
    return out


def downstream_reservoirs(plant):
    out = set()
    for e in connections:
        if e["from"] == plant and e["from_type"] == "plant":
            if e["to_type"] == "reservoir":
                out.add(e["to"])
            elif e["to_type"] in {"river", "tunnel"}:
                for e2 in connections:
                    if e2["from"] == e["to"] and e2["to_type"] == "reservoir":
                        out.add(e2["to"])
    return sorted(out)


GENERATORS = sorted(
    {c.removeprefix("result_committed_").rsplit("_t", 1)[0]
     for c in target.columns if c.startswith("result_committed")}
)

FEEDERS, PLANT_OF, DOWNSTREAM, OUTLET = {}, {}, {}, {}
for gen in GENERATORS:
    plant = next(e["to"] for e in connections
                if e["from"] == gen and e["from_type"] == "generator" and e["to_type"] == "plant")
    PLANT_OF[gen] = plant
    FEEDERS[gen] = sorted(supplying_reservoirs(plant, "plant"))
    DOWNSTREAM[gen] = downstream_reservoirs(plant)
    OUTLET[plant] = float(topology["model"]["plant"][plant]["outlet_line"])

## The five candidate feature groups

All are generator-specific additions on top of the 193-column baseline
(186 shared columns + 7 `local_*` aggregates over the intake reservoirs):

| group | columns | what it encodes |
|---|---|---|
| spill risk | 2 | max intake fill fraction + a `> 0.95` flag: a nearly full reservoir must spill or run |
| gross head | 1 | reservoir level (from the `vol_head` curve, capacity weighted) minus the plant outlet line: more head = more energy per m3 |
| water-value spread | 1 | terminal water value of intake minus downstream reservoir: full upstream / empty downstream pushes towards running |
| inflow anomaly | 1 local + 18 global | inflow vs the 1958-2014 seasonal normal: a wet or dry week relative to climatology |
| trailing price | 2 | mean price of the last 30 days and price minus that: the price *level* vs recent context |

The intuition we started from: full reservoirs above a plant and low water
below it force the plant to be active, and the price sets how active the whole
operation must be. The baseline already carries most of this (initial fill
fractions, terminal values, price minus value); the question is whether these
sharper encodings add anything on top.

In [5]:
def lookup(frame, timestamps, columns=None):
    f = frame if columns is None else frame[columns]
    return f.reindex(pd.DatetimeIndex(timestamps)).to_numpy(dtype=np.float32)


def feature_matrix(cases):
    """The 186 shared columns: one row per (case, hour)."""
    starts = pd.DatetimeIndex(cases["starttime"])
    n = len(starts)
    ts = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h"
    )
    price = lookup(price_h, ts).reshape(n, HOURS)
    initial = lookup(vol_wide, starts, RESERVOIRS)
    terminal = lookup(wv_wide, starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflow = lookup(inf_wide, ts).reshape(n, HOURS, -1)
    minflow = lookup(mf_wide, ts).reshape(n, HOURS, -1)
    minvol = lookup(mv_wide, ts).reshape(n, HOURS, -1)

    F = {}

    def add(name, v):
        v = np.asarray(v, dtype=np.float32)
        if v.shape == (n,):
            v = np.repeat(v, HOURS)
        elif v.shape == (n, HOURS):
            v = v.reshape(-1)
        F[name] = v

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", ts.hour.to_numpy())
    add("day_of_week", ts.dayofweek.to_numpy())
    add("month", ts.month.to_numpy())
    add("doy_sin", np.sin(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))
    add("doy_cos", np.cos(2 * np.pi * ts.dayofyear.to_numpy() / 365.25))

    add("price", price)
    for stat, v in (("mean", price.mean(1)), ("std", price.std(1)),
                    ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", v)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, v in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(v, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    for off in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{off:+d}", price[:, np.clip(np.arange(HOURS) + off, 0, HOURS - 1)])

    for j, res in enumerate(RESERVOIRS):
        add(f"init_vol__{res}", initial[:, j])
        add(f"init_frac__{res}", initial[:, j] / CAPACITY[res])
        add(f"term_val__{res}", terminal[:, j])
        add(f"price_minus_val__{res}", price - terminal[:, j, None])

    for j, name in enumerate(INF_COLS):
        flow = inflow[:, :, j]
        cum = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cum_Mm3__{name}", cum)
        add(f"inflow_rem_Mm3__{name}", cum[:, -1:] - cum)

    for j, name in enumerate(mf_wide.columns):
        add(f"minflow__{name}", minflow[:, :, j])
        add(f"minflow_week_max__{name}", minflow[:, :, j].max(1))
    for j, name in enumerate(mv_wide.columns):
        add(f"minvol__{name}", minvol[:, :, j])
        add(f"minvol_week_max__{name}", minvol[:, :, j].max(1))
        add(f"init_margin_to_minvol__{name}", initial[:, RESERVOIRS.index(name), None] - minvol[:, :, j])

    return pd.DataFrame(F, dtype=np.float32), starts, ts, price, initial, terminal, inflow

In [6]:
def local_features(starts, initial, terminal, inflow, price, anom, gen, groups):
    """The 7 baseline local columns plus any requested candidate groups,
    as an (n*168, k) array."""
    feeders = FEEDERS[gen]
    fcols = [INF_COLS.index(r) for r in feeders]
    fidx = [RESERVOIRS.index(r) for r in feeders]
    cap = np.array([CAPACITY[r] for r in feeders], dtype=np.float32)
    capsum = float(cap.sum())
    total_init = initial[:, fidx].sum(1)
    value = (terminal[:, fidx] * cap).sum(1) / capsum
    cum = inflow[:, :, fcols].sum(2).cumsum(1) * (3600 / 1e6)
    rem = cum[:, -1:] - cum

    cols = [
        np.repeat(total_init / capsum, HOURS),
        np.repeat(value, HOURS),
        price.reshape(-1) - np.repeat(value, HOURS),
        inflow[:, :, fcols].sum(2).reshape(-1),
        (cum / capsum).reshape(-1),
        (rem / capsum).reshape(-1),
        ((total_init[:, None] + cum + rem) / capsum).reshape(-1),
    ]

    if "spill" in groups:
        frac = initial[:, fidx] / cap
        max_frac = frac.max(1)
        cols.append(np.repeat(max_frac, HOURS))
        cols.append(np.repeat((max_frac > 0.95).astype(np.float32), HOURS))

    if "head" in groups:
        levels = np.stack([
            np.interp(initial[:, j],
                      np.asarray(topology["model"]["reservoir"][r]["vol_head"]["x"]),
                      np.asarray(topology["model"]["reservoir"][r]["vol_head"]["y"]))
            for j, r in zip(fidx, feeders)
        ], axis=1)
        level = (levels * cap).sum(1) / cap.sum()
        cols.append(np.repeat(level - OUTLET[PLANT_OF[gen]], HOURS))

    if "spread" in groups:
        down = DOWNSTREAM[gen]
        if down:
            didx = [RESERVOIRS.index(r) for r in down]
            dcap = np.array([CAPACITY[r] for r in down], dtype=np.float32)
            dval = (terminal[:, didx] * dcap).sum(1) / dcap.sum()
            spread = value - dval
        else:
            spread = np.zeros(len(starts), dtype=np.float32)
        cols.append(np.repeat(spread, HOURS))

    if "anom" in groups:
        cols.append(anom[:, :, fcols].sum(2).reshape(-1))

    return np.stack(cols, axis=1).astype(np.float32)

In [7]:
# seasonal normal per source from the long pre-2015 history
CLIM = inf_wide[:"2014"].groupby(inf_wide[:"2014"].index.dayofyear).mean()


def global_columns(starts, price, anom):
    """Candidate columns shared by all generators: per-source inflow anomaly
    (hourly) and trailing 30-day price context (per case)."""
    cols = {f"inflow_anom__{name}": anom[:, :, j].reshape(-1)
            for j, name in enumerate(INF_COLS)}
    trail = price_h.shift(1).rolling(720, min_periods=24).mean().reindex(starts)
    trail = trail.to_numpy(dtype=np.float32)
    trail = np.where(np.isfinite(trail), trail, price.mean(1))  # first 2015 cases
    cols["trailing_30d_mean"] = np.repeat(trail, HOURS)
    cols["price_minus_trailing_30d"] = price.reshape(-1) - np.repeat(trail, HOURS)
    return cols

## Two early-stopping protocols

- **`val`** (the old way): `eval_set=[(X_val, y_val)]`. The number of trees is
  chosen by validation AUC. This leaks the validation set into model selection,
  and it is *unstable*: a new feature can shift the AUC-vs-trees curve so that
  a model stops at 1-5 trees and collapses. Deltas of ±0.02 between feature
  sets under this protocol are not trustworthy.
- **`train_tail`** (the fix): `eval_set` is the last 90 days of the *train*
  period. The validation set is only ever scored, never fitted to in any way.

Both protocols are run below so the instability can be seen side by side.

In [8]:
def labels_for(cases, gen):
    cols = [f"result_committed_{gen}_t{t}" for t in range(HOURS)]
    return cases[cols].to_numpy(dtype=np.uint8).reshape(-1)


target["starttime"] = pd.to_datetime(target["starttime"], utc=True, format="mixed")

# step name -> (local groups, use global inflow anomaly, use trailing price)
STEPS = {
    "baseline":     ([],                         False, False),
    "+spill":       (["spill"],                  False, False),
    "+head":        (["head"],                   False, False),
    "+spread":      (["spread"],                 False, False),
    "+anom_local":  (["anom"],                   False, False),
    "+anom_global": ([],                         True,  False),
    "+trailing":    ([],                         False, True),
    "+head+spread": (["head", "spread"],         False, False),
    "+all":         (["spill", "head", "spread", "anom"], True, True),
}

EXPERIMENTS = (
    [(2022, "val", s) for s in
        ["baseline", "+spill", "+head", "+spread", "+anom_local", "+anom_global", "+trailing"]]
    + [(2022, "train_tail", s) for s in
        ["baseline", "+spill", "+head", "+spread", "+anom_local", "+anom_global",
         "+trailing", "+head+spread", "+all"]]
    + [(2021, "train_tail", s) for s in
        ["baseline", "+head", "+spread", "+head+spread", "+all"]]
    + [(2021, "val", s) for s in ["baseline", "+head"]]
)

In [9]:
def year_blocks(year):
    """Feature matrices for train / stop / val of one validation year."""
    split = pd.Timestamp(f"{year}-01-01", tz="UTC")
    train_cases = target[target["starttime"] < split].reset_index(drop=True)
    stop_cases = train_cases[
        train_cases["starttime"] >= split - pd.Timedelta(days=90)
    ].reset_index(drop=True)
    val_cases = target[target["starttime"] >= split].reset_index(drop=True)

    blocks = {}
    for name, cases in (("tr", train_cases), ("st", stop_cases), ("va", val_cases)):
        F, starts, ts, price, initial, terminal, inflow = feature_matrix(cases)
        anom = inflow - CLIM.reindex(ts.dayofyear.to_numpy()).to_numpy().reshape(
            len(starts), HOURS, -1)
        gcols = global_columns(starts, price, anom)
        blocks[name] = (F, starts, price, initial, terminal, inflow, anom, gcols)
    return train_cases, stop_cases, val_cases, blocks

In [10]:
t0 = time.perf_counter()
rows, cache = [], {}

for year, protocol, step_name in EXPERIMENTS:
    if year not in cache:
        cache[year] = year_blocks(year)
    train_cases, stop_cases, val_cases, blocks = cache[year]
    groups, use_anom, use_trail = STEPS[step_name]

    preds = {}
    for gen in GENERATORS:
        ytr = labels_for(train_cases, gen)
        yva = labels_for(val_cases, gen)

        parts = {}
        for name in ("tr", "st", "va"):
            F, starts, price, initial, terminal, inflow, anom, gcols = blocks[name]
            cols = [F.to_numpy(dtype=np.float32),
                    local_features(starts, initial, terminal, inflow, price, anom, gen, groups)]
            if use_anom:
                cols.append(np.stack([gcols[f"inflow_anom__{n}"] for n in INF_COLS], axis=1))
            if use_trail:
                cols.append(np.stack(
                    [gcols["trailing_30d_mean"], gcols["price_minus_trailing_30d"]], axis=1))
            parts[name] = np.hstack(cols)

        if np.unique(ytr).size == 1:
            preds[gen] = np.full(len(yva), float(ytr[0]), dtype=np.float32)
            continue

        yst = labels_for(stop_cases, gen)
        eval_set = [(parts["va"], yva)] if protocol == "val" else [(parts["st"], yst)]
        clf = xgb.XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=7,
            subsample=0.8, colsample_bytree=0.8, tree_method="hist",
            eval_metric="auc", early_stopping_rounds=25, random_state=0,
        )
        clf.fit(parts["tr"], ytr, eval_set=eval_set, verbose=False)
        preds[gen] = clf.predict_proba(parts["va"])[:, 1].astype(np.float32)

    y_all = np.concatenate([labels_for(val_cases, g) for g in GENERATORS])
    p_all = np.concatenate([preds[g] for g in GENERATORS])
    micro = roc_auc_score(y_all, p_all)
    rows.append({"year": year, "protocol": protocol, "step": step_name, "micro_auc": micro})
    print(f"[{time.perf_counter()-t0:6.0f}s] {year} {protocol:10s} {step_name:12s} "
          f"micro AUC={micro:.4f}", flush=True)

results = pd.DataFrame(rows)

[   111s] 2022 val        baseline     micro AUC=0.9686
[   229s] 2022 val        +spill       micro AUC=0.9526
[   363s] 2022 val        +head        micro AUC=0.9804
[   499s] 2022 val        +spread      micro AUC=0.9730
[   628s] 2022 val        +anom_local  micro AUC=0.9722
[   777s] 2022 val        +anom_global micro AUC=0.9747
[   900s] 2022 val        +trailing    micro AUC=0.9657
[  1100s] 2022 train_tail baseline     micro AUC=0.9803
[  1304s] 2022 train_tail +spill       micro AUC=0.9811
[  1513s] 2022 train_tail +head        micro AUC=0.9807
[  1714s] 2022 train_tail +spread      micro AUC=0.9804
[  1921s] 2022 train_tail +anom_local  micro AUC=0.9810
[  2129s] 2022 train_tail +anom_global micro AUC=0.9791
[  2335s] 2022 train_tail +trailing    micro AUC=0.9806
[  2540s] 2022 train_tail +head+spread micro AUC=0.9804
[  2769s] 2022 train_tail +all         micro AUC=0.9815
[  2951s] 2021 train_tail baseline     micro AUC=0.9694
[  3133s] 2021 train_tail +head        micro AUC

In [11]:
results.pivot_table(index="step", columns=["protocol", "year"], values="micro_auc").round(4)

protocol     train_tail             val        
year               2021    2022    2021    2022
step                                           
+all             0.9694  0.9815     NaN     NaN
+anom_global        NaN  0.9791     NaN  0.9747
+anom_local         NaN  0.9810     NaN  0.9722
+head            0.9692  0.9807  0.9652  0.9804
+head+spread     0.9691  0.9804     NaN     NaN
+spill              NaN  0.9811     NaN  0.9526
+spread          0.9689  0.9804     NaN  0.9730
+trailing           NaN  0.9806     NaN  0.9657
baseline         0.9694  0.9803  0.9503  0.9686

## What we measured

Full grid results (identical models, `random_state=0`, so these replicate
exactly):

**`val` protocol (early stopping on the validation set):**

| step | val 2021 | val 2022 |
|---|---|---|
| baseline | 0.9503 | 0.9686 |
| +spill | | 0.9526 |
| +head | 0.9652 | 0.9804 |
| +spread | | 0.9730 |
| +anom_local | | 0.9722 |
| +anom_global | | 0.9747 |
| +trailing | | 0.9657 |

**`train_tail` protocol (early stopping on the last 90 days of train):**

| step | val 2021 | val 2022 |
|---|---|---|
| baseline | 0.9694 | 0.9803 |
| +spill | | 0.9811 |
| +head | 0.9692 | 0.9807 |
| +spread | 0.9689 | 0.9804 |
| +anom_local | | 0.9810 |
| +anom_global | | 0.9791 |
| +trailing | | 0.9806 |
| +head+spread | 0.9691 | 0.9804 |
| +all | 0.9694 | 0.9815 |

## Takeaways

- Under the clean protocol **every feature-group delta is within ±0.0012** -
  noise. No group is worth adding. `+all` is nominally best on 2022
  (+0.0012) but exactly neutral on 2021, so it is not a real effect either.
- The `val` protocol swings (spill -0.016, head +0.012 on 2022) are roughly
  10x larger than anything real: they come from degenerate early stops, not
  from the features' information content.
- The protocol fix itself is the biggest measurable win of this whole
  search: **+0.012 on 2022 and +0.019 on 2021**, for zero extra features.
- The baseline feature set already carries the physical signal (fill
  fractions, terminal water values, price minus value, cumulative inflow).
  Sharper encodings of the same physics (head, spread, spill flags,
  climatology anomalies, trailing price) added nothing for XGBoost.